# NOGINX Model Retraining Notebook

This notebook recreates the full retraining pipeline and adds optional advanced anomaly features.

## What this notebook does
- Load `nginx_logs.csv`
- Build baseline features used by your API
- Build advanced behavioral/security features
- Encode categorical fields
- Train `IsolationForest`
- Save model and artifacts (`model.pkl`, `encoder_mappings.json`, `features.csv`)

In [1]:
from pathlib import Path
import json
import re

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import LabelEncoder

DATA_DIR = Path('.')
INPUT_CSV = DATA_DIR / 'nginx_logs.csv'

# Safer default: do not overwrite production model by mistake
OUTPUT_MODEL = DATA_DIR / 'model_retrained.pkl'
OUTPUT_ENCODERS = DATA_DIR / 'encoder_mappings_retrained.json'
OUTPUT_FEATURES = DATA_DIR / 'features_retrained.csv'

RANDOM_STATE = 42
CONTAMINATION = 'auto'

# Keep False for current API compatibility (fastapi expects 6 features)
USE_ADVANCED_FEATURES = False

# Set True only when you intentionally want to replace existing files
OVERWRITE_EXISTING_FILES = False

In [2]:
# Load dataset produced by parse_logs.py
# Expected columns: ip, timestamp, method, url, http_version, status, size, user_agent

df = pd.read_csv(INPUT_CSV)
required = {'ip', 'timestamp', 'method', 'url', 'status', 'size', 'user_agent'}
missing = required.difference(df.columns)
if missing:
    raise ValueError(f'Missing required columns: {sorted(missing)}')

# Parse time and normalize primitives
df['dt'] = pd.to_datetime(df['timestamp'], format='%d/%b/%Y:%H:%M:%S %z', errors='coerce')
df['dt'] = df['dt'].fillna(pd.to_datetime(df['timestamp'].str.split().str[0], format='%d/%b/%Y:%H:%M:%S', errors='coerce'))
df['hour_of_day'] = df['dt'].dt.hour.fillna(0).astype(int)
df['path'] = df['url'].astype(str).str.split('?').str[0]
df['status'] = pd.to_numeric(df['status'], errors='coerce').fillna(0).astype(int)
df['size'] = pd.to_numeric(df['size'], errors='coerce').fillna(0).astype(int)

# Sort by IP + time so sequential features are meaningful
df = df.sort_values(['ip', 'dt']).reset_index(drop=True)

df[['ip', 'timestamp', 'method', 'url', 'status', 'size', 'user_agent', 'hour_of_day', 'path']].head()

,ip,timestamp,method,url,status,size,user_agent,hour_of_day,path
0,10.0.0.10,24/May/2015:12:05:44 +0000,GET,/downloads/product_2,304,0,Debian APT-HTTP/1.3 (0.8.16~exp12ubuntu10.7),12,/downloads/product_2
1,10.0.0.10,24/May/2015:13:05:18 +0000,GET,/downloads/product_1,304,0,Debian APT-HTTP/1.3 (0.8.16~exp12ubuntu10.7),13,/downloads/product_1
2,10.0.0.10,24/May/2015:13:05:23 +0000,GET,/downloads/product_2,304,0,Debian APT-HTTP/1.3 (0.8.16~exp12ubuntu10.7),13,/downloads/product_2
3,10.0.0.10,24/May/2015:13:05:25 +0000,GET,/downloads/product_2,304,0,Debian APT-HTTP/1.3 (0.8.16~exp12ubuntu10.7),13,/downloads/product_2
4,10.0.0.10,24/May/2015:13:05:29 +0000,GET,/downloads/product_1,304,0,Debian APT-HTTP/1.3 (0.8.16~exp12ubuntu10.7),13,/downloads/product_1


In [4]:
# Build advanced features in one place (restart-safe)

def add_advanced_features(df: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    df = df.copy()
    df = df.sort_values(['ip', 'dt']).reset_index(drop=True)

    suspicious_pattern = r"(?:sql|union|select|drop|\.\./|<script|wp-login|admin|etc/passwd)"

    # Request rate proxy based on consecutive request gaps
    df['time_diff'] = df.groupby('ip')['dt'].diff().dt.total_seconds()
    df['req_per_min'] = (60.0 / df['time_diff']).replace([np.inf, -np.inf], 0.0)
    df['req_per_min'] = df['req_per_min'].fillna(1.0).clip(upper=1000.0)

    # Failure features
    df['is_4xx'] = ((df['status'] >= 400) & (df['status'] < 500)).astype(int)
    df['is_5xx'] = (df['status'] >= 500).astype(int)
    df['is_fail'] = ((df['status'] >= 400) & (df['status'] < 600)).astype(int)
    df['fail_rate_1min'] = df.groupby('ip')['is_fail'].transform(
        lambda x: x.rolling(window=10, min_periods=1).mean()
    )

    # Probing/scanning proxy
    df['unique_paths_1min'] = df.groupby('ip')['path'].transform(
        lambda x: x.rolling(window=10, min_periods=1).apply(lambda y: len(set(y)), raw=False)
    )

    # Session/time behavior
    df['prev_dt'] = df.groupby('ip')['dt'].shift(1)
    df['time_gap_sec'] = (df['dt'] - df['prev_dt']).dt.total_seconds().fillna(9999).clip(lower=0)
    first_seen = df.groupby('ip')['dt'].transform('min')
    df['session_duration_sec'] = (df['dt'] - first_seen).dt.total_seconds().fillna(0).clip(lower=0)

    # Method distribution
    df['is_get'] = (df['method'].astype(str).str.upper() == 'GET').astype(int)
    df['running_get_ratio'] = (
        df.groupby('ip')['is_get'].expanding().mean().reset_index(level=0, drop=True)
    ).fillna(0.0)

    # URL and security features
    df['url_len'] = df['url'].astype(str).str.len().astype(int)
    df['query_len'] = (
        df['url'].astype(str).str.extract(r'\?(.*)$', expand=False).fillna('').str.len().astype(int)
    )
    df['has_suspicious_keyword'] = df['url'].astype(str).str.contains(
        suspicious_pattern, na=False, regex=True
    ).astype(int)

    sensitive_paths = ('/admin', '/wp-login', '/login', '/config', '/.git', '/phpmyadmin')
    df['is_sensitive_path'] = df['path'].astype(str).str.contains(
        '|'.join(map(re.escape, sensitive_paths)), case=False, regex=True
    ).astype(int)

    # Response size spikes
    ip_size_mean = df.groupby('ip')['size'].transform('mean')
    ip_size_std = df.groupby('ip')['size'].transform('std').replace(0, np.nan)
    df['size_spike_score'] = ((df['size'] - ip_size_mean) / ip_size_std).fillna(0.0)

    advanced_cols = [
        'req_per_min', 'is_4xx', 'is_5xx', 'is_fail', 'fail_rate_1min',
        'unique_paths_1min', 'time_gap_sec', 'session_duration_sec',
        'running_get_ratio', 'url_len', 'query_len',
        'has_suspicious_keyword', 'is_sensitive_path', 'size_spike_score'
    ]
    return df, advanced_cols

In [5]:
# Encode categorical columns
method_encoder = LabelEncoder()
path_encoder = LabelEncoder()
ua_encoder = LabelEncoder()

df['method'] = method_encoder.fit_transform(df['method'].astype(str))
df['path'] = path_encoder.fit_transform(df['path'].astype(str))
df['user_agent'] = ua_encoder.fit_transform(df['user_agent'].astype(str))

encoder_mappings = {
    'method': {
        'classes': method_encoder.classes_.tolist(),
        'mapping': {label: int(i) for i, label in enumerate(method_encoder.classes_)}
    },
    'path': {
        'classes': path_encoder.classes_.tolist(),
        'mapping': {label: int(i) for i, label in enumerate(path_encoder.classes_)}
    },
    'user_agent': {
        'classes': ua_encoder.classes_.tolist(),
        'mapping': {label: int(i) for i, label in enumerate(ua_encoder.classes_)}
    }
}

baseline_cols = ['status', 'size', 'method', 'path', 'user_agent', 'hour_of_day']

if USE_ADVANCED_FEATURES:
    df, advanced_cols = add_advanced_features(df)
    feature_columns = baseline_cols + advanced_cols
else:
    feature_columns = baseline_cols

X_train = df[feature_columns].copy()
print(f'Rows: {len(X_train)} | Cols: {len(feature_columns)}')
feature_columns

Rows: 51412 | Cols: 6


['status', 'size', 'method', 'path', 'user_agent', 'hour_of_day']

In [6]:
# Train and save
model = IsolationForest(contamination=CONTAMINATION, random_state=RANDOM_STATE)
model.fit(X_train)

for out_path in [OUTPUT_MODEL, OUTPUT_ENCODERS, OUTPUT_FEATURES]:
    if out_path.exists() and not OVERWRITE_EXISTING_FILES:
        raise FileExistsError(
            f'{out_path} already exists. Set OVERWRITE_EXISTING_FILES=True or change output path.'
        )

joblib.dump(model, OUTPUT_MODEL)
with open(OUTPUT_ENCODERS, 'w', encoding='utf-8') as f:
    json.dump(encoder_mappings, f, indent=4)
X_train.to_csv(OUTPUT_FEATURES, index=False)

print(f'[OK] model saved: {OUTPUT_MODEL.resolve()}')
print(f'[OK] encoders saved: {OUTPUT_ENCODERS.resolve()}')
print(f'[OK] features saved: {OUTPUT_FEATURES.resolve()}')
print(f'[OK] training rows: {len(X_train)}')
print(f'[OK] training columns: {len(feature_columns)}')

[OK] model saved: E:\Krish\My_Learnings\Projects\noginx\data\model_retrained.pkl
[OK] encoders saved: E:\Krish\My_Learnings\Projects\noginx\data\encoder_mappings_retrained.json
[OK] features saved: E:\Krish\My_Learnings\Projects\noginx\data\features_retrained.csv
[OK] training rows: 51412
[OK] training columns: 6


## Metrics (precision / recall / F2 / accuracy)

This project uses **Isolation Forest (unsupervised)**, so there are no true labels.

To still show the numbers, this notebook computes **heuristic “anomaly labels”** from common attack patterns in the URL/status/size. Treat these metrics as *proxy metrics*, not real ground-truth accuracy.


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, fbeta_score

# Use a sample to keep evaluation fast
EVAL_SAMPLE_ROWS = 200000
if len(df) > EVAL_SAMPLE_ROWS:
    df_eval = df.sample(n=EVAL_SAMPLE_ROWS, random_state=RANDOM_STATE)
else:
    df_eval = df.copy()

# Heuristic labels: treat these patterns as “likely anomalies”
suspicious_pattern = r"(?:sql|union|select|drop|\.\./|<script|wp-login|admin|etc/passwd)"

sensitive_paths = (
    '/admin', '/wp-login', '/login', '/config', '/.git', '/phpmyadmin'
)

failed_status = (df_eval['status'] >= 400).astype(int)
suspicious_url = df_eval['url'].astype(str).str.contains(suspicious_pattern, na=False, regex=True).astype(int)
sensitive_path = df_eval['url'].astype(str).str.contains('|'.join(map(re.escape, sensitive_paths)), case=False, regex=True).astype(int)

# Big response sizes can indicate leaks/abuse
size_threshold = df_eval['size'].quantile(0.99)
high_size = (df_eval['size'] > size_threshold).astype(int)

y_true = ((failed_status + suspicious_url + sensitive_path + high_size) >= 1).astype(int)

# Model prediction (IsolationForest uses -1 for anomalies)
X_eval = df_eval[feature_columns].copy()
preds = model.predict(X_eval)
y_pred = (preds == -1).astype(int)

print('--- Proxy Evaluation (heuristic labels) ---')
print('rows evaluated:', len(df_eval))
print('precision:', precision_score(y_true, y_pred, zero_division=0))
print('recall:', recall_score(y_true, y_pred, zero_division=0))
print('f2_score:', fbeta_score(y_true, y_pred, beta=2, zero_division=0))
print('accuracy:', accuracy_score(y_true, y_pred))

# Helpful quick view of how many points were predicted as anomalies
print('predicted anomalies:', int(y_pred.sum()))


## Integration + overwrite note

- Current API expects 6 features: `status, size, method, path, user_agent, hour_of_day`.
- Keep `USE_ADVANCED_FEATURES = False` for immediate API compatibility.
- If `USE_ADVANCED_FEATURES = True`, your inference layer must compute and send all advanced columns too.

### Does running this overwrite `data/model.pkl`?
- In this cleaned notebook, default output is `model_retrained.pkl`, so `data/model.pkl` is **not** changed automatically.
- `data/model.pkl` changes only if you set `OUTPUT_MODEL = DATA_DIR / 'model.pkl'` and `OVERWRITE_EXISTING_FILES = True`.